In [ ]:
%%sql -r dataframe_1
CREATE WAREHOUSE IF NOT EXISTS RETAIL_WH
  WAREHOUSE_SIZE = 'XSMALL'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE
  INITIALLY_SUSPENDED = TRUE;

In [ ]:
%%sql -r dataframe_3
CREATE DATABASE IF NOT EXISTS RETAIL_DB;

In [ ]:
%%sql -r dataframe_4
CREATE SCHEMA IF NOT EXISTS RETAIL_DB.CUSTOMER_SCD;

In [ ]:
%%sql -r dataframe_2
USE WAREHOUSE RETAIL_WH;


In [ ]:
%%sql -r dataframe_5
USE DATABASE RETAIL_DB;

In [ ]:
%%sql -r dataframe_6

USE SCHEMA CUSTOMER_SCD;

In [ ]:
%%sql -r dataframe_7
CREATE OR REPLACE FILE FORMAT CSV_FF
  TYPE = CSV
  FIELD_DELIMITER = ','
  SKIP_HEADER = 1
  FIELD_OPTIONALLY_ENCLOSED_BY = '"';



In [ ]:
%%sql -r dataframe_8
CREATE OR REPLACE STAGE CUSTOMER_STAGE
  FILE_FORMAT = CSV_FF;

In [ ]:
%%sql -r dataframe_9
CREATE OR REPLACE TABLE STG_CUSTOMERS_INITIAL (
  CUSTOMER_ID    NUMBER,
  CUSTOMER_NAME  VARCHAR(100),
  CITY           VARCHAR(50),
  STATE          VARCHAR(50),
  MEMBERSHIP     VARCHAR(30),
  SEGMENT        VARCHAR(30)
);

In [ ]:
%%sql -r dataframe_10
COPY INTO STG_CUSTOMERS_INITIAL
FROM @CUSTOMER_STAGE/customers_initial.csv;

In [ ]:
%%sql -r dataframe_11
CREATE OR REPLACE TABLE STG_CUSTOMER_UPDATES (
  CUSTOMER_ID    NUMBER,
  CUSTOMER_NAME  VARCHAR(100),
  CITY           VARCHAR(50),
  STATE          VARCHAR(50),
  MEMBERSHIP     VARCHAR(30),
  SEGMENT        VARCHAR(30),
  EFFECTIVE_DATE DATE
);

In [ ]:
%%sql -r dataframe_12
COPY INTO STG_CUSTOMER_UPDATES
FROM @CUSTOMER_STAGE/customer_updates.csv;

In [ ]:
%%sql -r dataframe_13
CREATE OR REPLACE TABLE CUSTOMER_SCD1 (
  CUSTOMER_KEY   NUMBER AUTOINCREMENT START 1 INCREMENT 1,
  CUSTOMER_ID    NUMBER,
  CUSTOMER_NAME  VARCHAR(100),
  CITY           VARCHAR(50),
  STATE          VARCHAR(50),
  MEMBERSHIP     VARCHAR(30),
  SEGMENT        VARCHAR(30)
);

In [ ]:
%%sql -r dataframe_14
INSERT INTO CUSTOMER_SCD1 (CUSTOMER_ID, CUSTOMER_NAME, CITY, STATE, MEMBERSHIP, SEGMENT)
SELECT CUSTOMER_ID, CUSTOMER_NAME, CITY, STATE, MEMBERSHIP, SEGMENT
FROM STG_CUSTOMERS_INITIAL;

In [ ]:
%%sql -r dataframe_15
MERGE INTO CUSTOMER_SCD1 AS tgt
USING STG_CUSTOMER_UPDATES AS src
  ON tgt.CUSTOMER_ID = src.CUSTOMER_ID
WHEN MATCHED THEN UPDATE SET
  tgt.CITY       = src.CITY,
  tgt.STATE      = src.STATE,
  tgt.MEMBERSHIP = src.MEMBERSHIP,
  tgt.SEGMENT    = src.SEGMENT;

In [ ]:
%%sql -r dataframe_16
SELECT CUSTOMER_ID, CUSTOMER_NAME, CITY, STATE, MEMBERSHIP, SEGMENT
FROM CUSTOMER_SCD1
ORDER BY CUSTOMER_ID;

In [ ]:
%%sql -r dataframe_17
SELECT CUSTOMER_ID, CITY, STATE, MEMBERSHIP
FROM CUSTOMER_SCD1
WHERE CUSTOMER_ID = 101;

In [ ]:
%%sql -r dataframe_18
CREATE OR REPLACE TABLE CUSTOMER_SCD2 (
  CUSTOMER_KEY    NUMBER AUTOINCREMENT START 1 INCREMENT 1,
  CUSTOMER_ID     NUMBER,
  CUSTOMER_NAME   VARCHAR(100),
  CITY            VARCHAR(50),
  STATE           VARCHAR(50),
  MEMBERSHIP      VARCHAR(30),
  SEGMENT         VARCHAR(30),
  EFFECTIVE_DATE  DATE,
  EXPIRY_DATE     DATE,
  IS_CURRENT      BOOLEAN
);

In [ ]:
%%sql -r dataframe_19
INSERT INTO CUSTOMER_SCD2
  (CUSTOMER_ID, CUSTOMER_NAME, CITY, STATE, MEMBERSHIP, SEGMENT,
   EFFECTIVE_DATE, EXPIRY_DATE, IS_CURRENT)
SELECT
  CUSTOMER_ID, CUSTOMER_NAME, CITY, STATE, MEMBERSHIP, SEGMENT,
  '2026-01-01', '9999-12-31', TRUE
FROM STG_CUSTOMERS_INITIAL;

In [ ]:
%%sql -r dataframe_20
SELECT COUNT(*) AS TOTAL_RECORDS FROM CUSTOMER_SCD2;

In [ ]:
%%sql -r dataframe_21
SELECT COUNT(*) AS CURRENT_RECORDS FROM CUSTOMER_SCD2 WHERE IS_CURRENT = TRUE;

In [ ]:
%%sql -r dataframe_22
UPDATE CUSTOMER_SCD2 tgt
SET EXPIRY_DATE = DATEADD(day, -1, src.EFFECTIVE_DATE),
    IS_CURRENT  = FALSE
FROM STG_CUSTOMER_UPDATES src
WHERE tgt.CUSTOMER_ID = src.CUSTOMER_ID
  AND tgt.IS_CURRENT  = TRUE;

In [ ]:
%%sql -r dataframe_23
INSERT INTO CUSTOMER_SCD2
  (CUSTOMER_ID, CUSTOMER_NAME, CITY, STATE, MEMBERSHIP, SEGMENT,
   EFFECTIVE_DATE, EXPIRY_DATE, IS_CURRENT)
SELECT
  CUSTOMER_ID, CUSTOMER_NAME, CITY, STATE, MEMBERSHIP, SEGMENT,
  EFFECTIVE_DATE, '9999-12-31', TRUE
FROM STG_CUSTOMER_UPDATES;

In [ ]:
%%sql -r dataframe_24
SELECT CUSTOMER_ID, CITY, MEMBERSHIP, EFFECTIVE_DATE, EXPIRY_DATE, IS_CURRENT
FROM CUSTOMER_SCD2 WHERE CUSTOMER_ID = 101 ORDER BY EFFECTIVE_DATE;

In [ ]:
%%sql -r dataframe_25
SELECT CUSTOMER_ID, CITY, MEMBERSHIP, EFFECTIVE_DATE, EXPIRY_DATE, IS_CURRENT
FROM CUSTOMER_SCD2 WHERE CUSTOMER_ID = 103 ORDER BY EFFECTIVE_DATE;


In [ ]:
%%sql -r dataframe_26
SELECT CUSTOMER_ID, CITY, MEMBERSHIP, EFFECTIVE_DATE, EXPIRY_DATE, IS_CURRENT
FROM CUSTOMER_SCD2 WHERE CUSTOMER_ID = 104 ORDER BY EFFECTIVE_DATE;

In [ ]:
%%sql -r dataframe_27
SELECT CUSTOMER_ID, CUSTOMER_NAME, CITY, STATE, MEMBERSHIP,
       EFFECTIVE_DATE, EXPIRY_DATE, IS_CURRENT
FROM CUSTOMER_SCD2
ORDER BY CUSTOMER_ID, EFFECTIVE_DATE;

In [ ]:
%%sql -r dataframe_28
SELECT CUSTOMER_ID, CUSTOMER_NAME, CITY, STATE, MEMBERSHIP, SEGMENT
FROM CUSTOMER_SCD2
WHERE IS_CURRENT = TRUE
ORDER BY CUSTOMER_ID;

In [ ]:
%%sql -r dataframe_29
SELECT CUSTOMER_ID, CUSTOMER_NAME, MEMBERSHIP, CITY, EFFECTIVE_DATE, EXPIRY_DATE
FROM CUSTOMER_SCD2
WHERE CUSTOMER_ID = 101
  AND '2026-03-15' BETWEEN EFFECTIVE_DATE AND EXPIRY_DATE;

In [ ]:
%%sql -r dataframe_30
SELECT COUNT(*) AS SCD_TYPE1_RECORD_COUNT FROM CUSTOMER_SCD1;

In [ ]:
%%sql -r dataframe_31
SELECT COUNT(*) AS SCD_TYPE2_RECORD_COUNT FROM CUSTOMER_SCD2;


In [ ]:
%%sql -r dataframe_32
SELECT COUNT(*) AS SCD_TYPE2_CURRENT_RECORD_COUNT
FROM CUSTOMER_SCD2 WHERE IS_CURRENT = TRUE;

In [ ]:
%%sql -r dataframe_33
SELECT COUNT(*) AS SCD_TYPE2_HISTORICAL_RECORD_COUNT
FROM CUSTOMER_SCD2 WHERE IS_CURRENT = FALSE;